# LAB 4 — Git Basics for a Python Project**Student:** Prabhas P Simha  **USN:** 24SSDS415036  **GitHub:** https://github.com/Pubby-15  **Repository:** https://github.com/Pubby-15/ADV-Lab-05This notebook follows the same sequence as the reference lab: configure Git, create/check project files, add `.gitignore`, stage and commit, connect and push to GitHub, update the Streamlit dashboard, create a feature branch, and prepare deployment.**Important:** The repository already exists. The first setup cell clones it so the notebook does not overwrite its Git history. Starter files are created only when missing. A GitHub token is requested through a hidden prompt and is never written into this notebook.

In [ ]:
# Step 1: Configure Git identity and clone the existing GitHub repositoryimport osimport subprocessfrom pathlib import Pathsubprocess.run(["git", "config", "--global", "user.name", "Prabhas P Simha"], check=True)git_email = input("Enter the email associated with your GitHub account: ").strip()if not git_email:    raise ValueError("Enter a valid GitHub-associated email.")subprocess.run(["git", "config", "--global", "user.email", git_email], check=True)GITHUB_USER = "Pubby-15"REPO_NAME = "ADV-Lab-05"REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"if not Path(".git").exists():    if Path(REPO_NAME).exists():        os.chdir(REPO_NAME)    else:        subprocess.run(["git", "clone", REPO_URL, REPO_NAME], check=True)        os.chdir(REPO_NAME)print("Working directory:", Path.cwd())subprocess.run(["git", "status"], check=True)

In [ ]:
# Step 2: Create/check the project files without overwriting existing repository workif not Path("app.py").exists():    Path("app.py").write_text("print('hello from my project')\n", encoding="utf-8")if not Path("requirements.txt").exists():    Path("requirements.txt").write_text("pandas\nmatplotlib\n", encoding="utf-8")if not Path("README.md").exists():    Path("README.md").write_text(        "# My Project\n\nA small project for the Git Basics lab.\n",        encoding="utf-8",    )print("Project files:", sorted(p.name for p in Path(".").iterdir()))

In [ ]:
# Step 3: Create the .gitignore file# These rules exclude temporary files, local data, and environment secrets.gitignore_content = """.ipynb_checkpoints/__pycache__/*.pyc.env.venv/venv/data/*.csv*.xlsx*.parquet"""Path(".gitignore").write_text(gitignore_content, encoding="utf-8")print(".gitignore created/updated")

In [ ]:
# Step 4: Initialize Git, stage files, and check repository statussubprocess.run(["git", "init", "-q"], check=True)subprocess.run(["git", "add", "."], check=True)subprocess.run(["git", "status"], check=True)

In [ ]:
# Step 5: Create the initial commit only when staged changes existstaged_changes = subprocess.run(["git", "diff", "--cached", "--quiet"], check=False)if staged_changes.returncode != 0:    subprocess.run(["git", "commit", "-m", "Initial commit: project skeleton"], check=True)else:    print("No new staged changes; the existing repository history is retained.")subprocess.run(["git", "log", "--oneline", "-5"], check=True)

In [ ]:
# Step 6: Connect the GitHub remote and push securely using a hidden PAT promptimport getpassimport tempfileremote_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"remotes = subprocess.run(["git", "remote"], capture_output=True, text=True, check=True).stdout.split()if "origin" in remotes:    subprocess.run(["git", "remote", "set-url", "origin", remote_url], check=True)else:    subprocess.run(["git", "remote", "add", "origin", remote_url], check=True)# Enter a Personal Access Token only when prompted. It is not saved in the notebook.github_pat = getpass.getpass("Paste your GitHub PAT (hidden): ")if not github_pat:    raise ValueError("A GitHub PAT is required for HTTPS authentication.")askpass = Path(tempfile.gettempdir()) / "git_askpass_lab4.sh"askpass.write_text(    """#!/bin/shcase "$1" in  *sername*) printf '%s\\n' "$GITHUB_USERNAME" ;;  *assword*) printf '%s\\n' "$GITHUB_TOKEN" ;;  *) exit 1 ;;esac""",    encoding="utf-8",)askpass.chmod(0o700)def git_push_secure(branch):    env = os.environ.copy()    env.update({        "GIT_ASKPASS": str(askpass),        "GIT_ASKPASS_REQUIRE": "force",        "GIT_TERMINAL_PROMPT": "0",        "GITHUB_USERNAME": GITHUB_USER,        "GITHUB_TOKEN": github_pat,    })    try:        subprocess.run(["git", "push", "-u", "origin", branch], env=env, check=True)    finally:        env.pop("GITHUB_TOKEN", None)subprocess.run(["git", "branch", "-M", "main"], check=True)git_push_secure("main")

In [ ]:
# Step 7: Update app.py with the Streamlit dashboard from the previous labapp_code = r'''import streamlit as stimport pandas as pdimport plotly.express as pxst.set_page_config(page_title="County Development Dashboard", layout="wide")@st.cache_datadef load_data():    return px.data.gapminder()df = load_data()st.title("County Development Dashboard")st.sidebar.header("Filters")years = sorted(df["year"].unique())selected_year = st.sidebar.slider(    "Year", int(min(years)), int(max(years)), int(max(years)), 5)continents = sorted(df["continent"].unique())selected_continents = st.sidebar.multiselect(    "Continent", continents, default=continents)filtered = df[(df["year"] == selected_year) & (df["continent"].isin(selected_continents))].copy()st.subheader("Key Metrics")col1, col2, col3 = st.columns(3)col1.metric("Countries", int(filtered["country"].nunique()))col2.metric(    "Median life expectancy",    f'{filtered["lifeExp"].median():.1f} yrs' if not filtered.empty else "0.0 yrs",)col3.metric(    "Total population",    f'{filtered["pop"].sum() / 1_000_000:.2f} B' if not filtered.empty else "0.00 B",)st.subheader("GDP per Capita vs Life Expectancy")if filtered.empty:    st.warning("No records match the selected filters.")else:    fig = px.scatter(        filtered,        x="gdpPercap",        y="lifeExp",        size="pop",        color="continent",        hover_name="country",        log_x=True,        size_max=55,        title=f"Development Overview — {selected_year}",        labels={"gdpPercap": "GDP per capita", "lifeExp": "Life expectancy", "pop": "Population"},    )    fig.update_layout(height=620)    st.plotly_chart(fig, use_container_width=True)st.caption("LAB 4 — Git Basics for a Python Project")'''Path("app.py").write_text(app_code, encoding="utf-8")print("app.py updated with the Streamlit dashboard")

In [ ]:
# Step 8: Stage, commit, and push the updated dashboardsubprocess.run(["git", "add", "app.py"], check=True)staged_changes = subprocess.run(["git", "diff", "--cached", "--quiet"], check=False)if staged_changes.returncode != 0:    subprocess.run(["git", "commit", "-m", "Update app.py"], check=True)else:    print("app.py already contains the dashboard version.")git_push_secure("main")

In [ ]:
# Step 9: View the commit history after updating app.pysubprocess.run(["git", "log", "--oneline"], check=True)

In [ ]:
# Step 10: View the current branchessubprocess.run(["git", "branch"], check=True)

In [ ]:
# Step 11: Create or switch to the feature branchbranch_name = "feature/experiment"subprocess.run(["git", "fetch", "origin"], check=True)local_branch = subprocess.run(    ["git", "branch", "--list", branch_name],    capture_output=True,    text=True,    check=True,).stdout.strip()remote_branch = subprocess.run(    ["git", "branch", "-r", "--list", f"origin/{branch_name}"],    capture_output=True,    text=True,    check=True,).stdout.strip()if local_branch:    subprocess.run(["git", "switch", branch_name], check=True)elif remote_branch:    subprocess.run(["git", "switch", "--track", f"origin/{branch_name}"], check=True)else:    subprocess.run(["git", "switch", "-c", branch_name], check=True)

In [ ]:
# Step 12: Verify the active feature branchsubprocess.run(["git", "branch"], check=True)

In [ ]:
# Step 13: Make a small change on the feature branchwith open("app.py", "r+", encoding="utf-8") as f:    content = f.read()    feature_note = "\n# Feature experiment branch update\n"    if "# Feature experiment branch update" not in content:        f.write(feature_note)print("Feature branch update added only once.")

In [ ]:
# Step 14: Display the last five lines of app.pyprint("\n".join(Path("app.py").read_text(encoding="utf-8").splitlines()[-5:]))

In [ ]:
# Step 15: Check the changed filessubprocess.run(["git", "status"], check=True)

In [ ]:
# Step 16: Stage the feature branch updatesubprocess.run(["git", "add", "app.py"], check=True)

In [ ]:
# Step 17: Commit the feature branch update if there are staged changesstaged_changes = subprocess.run(["git", "diff", "--cached", "--quiet"], check=False)if staged_changes.returncode != 0:    subprocess.run(["git", "commit", "-m", "Add experiment feature"], check=True)else:    print("No new feature-branch changes to commit.")

In [ ]:
# Step 18: Push the feature branch to GitHubgit_push_secure("feature/experiment")

In [ ]:
# Step 19: Switch back to the main branchsubprocess.run(["git", "switch", "main"], check=True)

In [ ]:
# Step 20: Check the current dependenciesprint(Path("requirements.txt").read_text(encoding="utf-8"))

In [ ]:
# Step 21: Update requirements.txt for the Streamlit dashboardPath("requirements.txt").write_text("streamlit\npandas\nplotly\n", encoding="utf-8")print("requirements.txt updated")

In [ ]:
# Step 22: Commit and push the deployment requirementssubprocess.run(["git", "add", "requirements.txt"], check=True)staged_changes = subprocess.run(["git", "diff", "--cached", "--quiet"], check=False)if staged_changes.returncode != 0:    subprocess.run(["git", "commit", "-m", "Update requirements for Streamlit"], check=True)else:    print("requirements.txt already contains the dashboard dependencies.")git_push_secure("main")# Remove the temporary askpass helper and clear the token variable after the pushes.askpass.unlink(missing_ok=True)github_pat = Noneprint("Git workflow complete. The Streamlit deployment is still a separate step.")

## Step 22: Prepare deployment on Streamlit Community Cloud1. Open [Streamlit Community Cloud](https://share.streamlit.io/).2. Sign in with the GitHub account **Pubby-15**.3. Select **Create app** and choose repository **Pubby-15/ADV-Lab-05**.4. Select branch **main** and main file **app.py**.5. Click **Deploy** and wait for dependency installation.6. Copy the generated public `.streamlit.app` address after deployment succeeds.**Current submission status:** The GitHub repository and application files are prepared. A live Streamlit deployment is still pending and should not be reported as completed until a working URL is generated.